In [0]:
import json
from pyspark.sql import functions as F

def get_widget(name: str, default: str) -> str:
    if "dbutils" not in globals():
        return default
    try:
        return dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default)
        return dbutils.widgets.get(name)

bucket = get_widget("s3_bucket", "de-e2e-413612133697-ap-southeast-1-an")
manifest_prefix = get_widget("s3_media_manifest_prefix", "lakehouse/landing/douyin/media_manifest/json")
media_manifest_uris_json = get_widget("media_manifest_uris_json", "[]")
media_manifest_uris = json.loads(media_manifest_uris_json or "[]")

manifest_root = f"s3://{bucket}/{manifest_prefix.strip('/')}/"
manifest_input = media_manifest_uris if media_manifest_uris else manifest_root
bronze_manifest_path = f"s3://{bucket}/lakehouse/bronze/douyin/media_manifest_raw_delta/"

print(f"Bronze media manifest input count: {len(media_manifest_uris) if media_manifest_uris else 'ALL'}")

raw_text_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .text(manifest_input)
    .select(
        F.col("_metadata.file_path").alias("source_file"),
        F.col("value").alias("raw_json_string"),
        F.current_timestamp().alias("bronze_ingested_at")
    )
)

parsed_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .json(manifest_input)
    .select(
        F.col("_metadata.file_path").alias("source_file"),
        F.col("pipeline"),
        F.col("source"),
        F.col("zone"),
        F.col("artifact"),
        F.col("niche"),
        F.col("account_id"),
        F.col("generated_at"),
        F.struct("*").alias("raw_struct")
    )
)

bronze_manifest_df = raw_text_df.join(parsed_df, on="source_file", how="inner")

if bronze_manifest_df.count() == 0:
    raise ValueError("No media manifest files were read for bronze")

display(bronze_manifest_df)


In [0]:
(
    bronze_manifest_df.write
    .format("delta")
    .mode("append")
    .save(bronze_manifest_path)
)

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS de_e2e.bronze.douyin_media_manifest_raw
            USING DELTA
            LOCATION '{bronze_manifest_path}'
          """)

In [0]:
%sql
SELECT source_file, account_id, artifact
FROM de_e2e.bronze.douyin_media_manifest_raw
LIMIT 20;